# Sampling and Bootstrap Distributions of Parameters

Now that we understand how moments are used to parameterize distributions, we can discuss different types of distributions that come up in working with random data and parameter estimates. In particular, we have introduced the general idea of the bootstrap distribution in {doc}`Section 5.6<bootstrap-dist-confidence-intervals.ipynb>` but at that time did not have the mathematical tools that we use for characterizing the distributions of random variables that we have available to us at this point in this book. 

## Sampling Distribution of an Estimator

Consider a scenario in which we have independent random variables $\mathbf{X} = \left[ X_0, X_1, \ldots, X_{n-1}\right]$ from a  distribution that is characterized by some parameter $\theta$. Let $\hat{\Theta}$ be an estimator for $\theta$; that is $\hat{\Theta}$ is some function $\hat{\Theta} = g(\mathbf{X})$, where $g()$ is chosen to make $\hat{\Theta}$ be a "good" (e.g. unbiased, low MSE) estimator for $\theta$. Then $\hat{\Theta}$ is itself a random variable and hence has some distribution that is generally different than the distribution of the $X_i$. The distribution of $\hat{\Theta}$ is called the *sampling distribution*:


````{card}
DEFINITION
^^^
```{glossary}
sampling distribution
    Given a vector of independent random variables $\mathbf{X}$ and a parameter estimator $\hat{\Theta}$, the *sampling distribution* characterizes the probability distribution of $\hat{\Theta}$.
```
````

This will be made much more clear with an example. Suppose we have 25 random values from some random distribution and we want to estimate the sampling distribution for the mean estimator. The function below takes as input a SciPy distribution object, creates $2^16$ values of the sample mean for samples of length 25, and then displays a 

In [36]:
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np


from matplotlib import animation
from matplotlib.animation import PillowWriter
from IPython.display import HTML


def sampling_dist(dist_obj, sample_length=25, samples=2**16):
  np.random.seed(45673376) # arbitray but want results to be consistent
  num_means = 2**16

  xvals = dist_obj.rvs(size=(num_means, sample_length))
  means=xvals.mean(axis=1)

  fig = plt.figure()
  fig.set_dpi(100)
  fig.set_size_inches(5, 4)




  ax = plt.axes(xlim=(np.min(means), np.max(means)), ylim=(0, 1))
  #plt.grid(axis='y')


  def animate(i):
    plt.clf()
    num=2*2**(i+1)

    if num <= 64:
      mybins=np.linspace(np.min(means), np.max(means), 10)
    elif num <= 1024:
      mybins=np.linspace(np.min(means), np.max(means), 20)
    else:
      mybins=np.linspace(np.min(means), np.max(means), 50)


    plt.hist(means[:num], bins=mybins, density=True);
    plt.title(f'# of means = {num}');

    return []



  anim = animation.FuncAnimation(fig, animate, 
                                 frames=int(np.log2(num_means/4))+1,
                                 interval=15,
                                 blit=False, repeat=False)

  #Comment this out and uncomment the anim.save() to write the GIF
  out = anim.to_jshtml(fps=2) 
  plt.close()
  print(np.min(means), np.max(means))
  return HTML(out)

In [23]:
U= stats.uniform(0,10)
sampling_dist(U)

2.66250088082409 7.654171298330203


Here is the result for the exponential random variable with mean 5:

In [27]:
E=stats.expon(scale=5)
sampling_dist(E)

1.8645007213223084 9.985991421964753


And here are the results for a chi-squared random variable with 6 degrees of freedom:

In [29]:
E=stats.chi2(df=6)
sampling_dist(E)

3.4682835754697057 9.65229322060737


Let's try a discrete distribution. Here is the sampling distribution for the mean of a binomial random variable with 20 trials and probability of success 0.2:

In [31]:
B = stats.binom(100, 0.2)
sampling_dist(B)

16.76 23.52


Even the sampling distribution for the mean of this Binomial random variable looks almost Normal (although it does have some "spikes" in the distribution that come from its discrete nature).

By the Central Limit Theorem, the average of a sequence of almost any time of independent random variables converges to the Normal distribution as the number of random variables being averaged goes to infinity. These results suggest that the sampling distribution of the mean is approximately Normal even if tens of random variables are averaged together. The sampling distribution will not be well approximated by the Normal for most distributions if the number of random variables being averaged is small. For instance, the average of two uniform random variables has a triangle-shaped density:

In [38]:
U= stats.uniform(0,10)
sampling_dist(U, sample_length=2)

0.026216512289113925 9.995033614282763


## Bootstrap Distribution of an Estimator